In [ ]:
%pip install pandas xgboost scikit-learn fastapi uvicorn

In [ ]:
import pandas as pd
import re

def charger_et_nettoyer_dataset(chemin_fichier):
    # Ignorer les lignes corrompues
    df_brut = pd.read_csv(chemin_fichier, on_bad_lines='skip')
    df_brut.columns = df_brut.columns.str.strip()
    
    df_clean = pd.DataFrame()
    
    # Extraction de la marque
    def trouver_marque(nom):
        nom_str = str(nom)
        if nom_str.startswith("G.Skill"): return "G.Skill"
        if nom_str.startswith("Silicon Power"): return "Silicon Power"
        return nom_str.split()[0] if nom_str.split() else "Unknown"
        
    df_clean['brand'] = df_brut['Name'].apply(trouver_marque)
    
    # Extraction de la génération et sa fréquence
    generation_speed = df_brut['Speed'].str.split('-', n=1).str
    df_clean['generation'] = generation_speed[0]
    df_clean['frequency_mhz'] = pd.to_numeric(generation_speed[1], errors='coerce').fillna(2133).astype(int)
    
    # Extraction de la capacité
    def trouver_capacite(nom):
        match = re.search(r'(\d+)\s*GB', str(nom))
        return int(match.group(1)) if match else 16
        
    df_clean['capacity_gb'] = df_brut['Name'].apply(trouver_capacite)
    
    # Nettoyage et conversion des prix
    df_clean['price'] = df_brut['Price'].astype(str).str.replace('$', '', regex=False).str.strip()
    df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce').fillna(0.0)
    
    df_clean['price_per_gb'] = df_brut['Price Per GB'].astype(str).str.replace('$', '', regex=False).str.strip()
    df_clean['price_per_gb'] = pd.to_numeric(df_clean['price_per_gb'], errors='coerce').fillna(0.0)
    
    # Création de la colonne numérique pour l'entraînement de l'IA
    df_clean['generation_num'] = df_clean['generation'].map({'DDR4': 4, 'DDR5': 5}).fillna(4).astype(int)
    
    return df_clean

In [ ]:
# Chargement et nettoyage du dataset
df = charger_et_nettoyer_dataset('ram_dataset.csv')

# Filtrage pour suppression des lignes avec un prix par Go nul ou négatif
df = df[df['price_per_gb'] > 0]

# Aperçu du nouveau dataset nettoyé
df.head()

In [ ]:
# On observe le prix moyen par Go selon la génération
df.groupby('generation')['price_per_gb'].mean()

In [ ]:
from xgboost import XGBRegressor
# Définition des fonctionnalités (X) et de la cible (y)
X = df[['capacity_gb', 'frequency_mhz', 'generation_num']]
y = df['price_per_gb']

# Initialisation et entraînement du modèle
modele_ram = XGBRegressor(n_estimators=50, learning_rate=0.1, max_depth=5)
modele_ram.fit(X, y)

print("✅ Modèle XGBoost entraîné avec succès !")

In [ ]:
# ZONE DE TEST :
generation_test = "DDR4"   # Choix : "DDR4" ou "DDR5"
frequence_test = 3200      # Fréquence en MHz
capacite_test = 32         # Capacité totale en Go
prix_vendeur_test = 120.0  # Le prix total affiché par le vendeur (en $)

# Logique de simulation de l'API
generation_texte = generation_test.upper()
erreur = False

# 🛑 Sécurités sur les fréquences
if generation_texte == 'DDR4' and (frequence_test > 4400 or frequence_test < 2133):
    print("❌ Configuration Impossible : La DDR4 doit être entre 2133 et 4400 MHz.")
    erreur = True
elif generation_texte == 'DDR5' and (frequence_test < 4800 or frequence_test > 8400):
    print("❌ Configuration Impossible : La DDR5 doit être entre 4800 et 8400 MHz.")
    erreur = True

if not erreur:
    # Encodage pour l'IA
    mapping_generation = {'DDR4': 4, 'DDR5': 5}
    gen_num = mapping_generation.get(generation_texte, 4)

    # Préparation de la ligne de données
    donnees_entree = pd.DataFrame([{
        'capacity_gb': capacite_test,
        'frequency_mhz': frequence_test,
        'generation_num': gen_num
    }])
    
    # Prédiction du prix par Go, puis calcul du prix total
    prix_par_gb_estime = float(modele_ram.predict(donnees_entree)[0])
    prix_ia = prix_par_gb_estime * capacite_test
    
    # Calcul de l'écart avec le vendeur
    ecart = (prix_ia - prix_vendeur_test) / prix_ia
    
    # Détermination du verdict
    if ecart >= 0.40:
        statut = "🔥 Excellent deal !"
    elif ecart <= -0.30:
        statut = "❌ Way too expensive."
    else:
        statut = "⚖️ Price within the market standard."
        
    # Affichage du rapport
    print("=== RAPPORT DE L'ANALYSEUR DE RAM ===")
    print(f"RAM testée          : {generation_test} {capacite_test}GB @ {frequence_test}MHz")
    print(f"Prix vendeur fourni : {prix_vendeur_test} $")
    print("-" * 37)
    print(f"Prix estimé par l'IA : {round(prix_ia, 2)} $ (soit {round(prix_par_gb_estime, 2)} $/Go)")
    print(f"Écart constaté      : {round(ecart * 100, 1)} %")
    print(f"Verdict             : {statut}")